# Next step: step 5a, the mass model
**What it adds:** the six values that shape the **unit galaxy** (`lnl`, `r_scale`, `m_halo`, `m_bulge`, `h_bulge`,
`h_disk`), a choice of `normalization`, and `sim.mass_profile(config)` to look at the galaxy's tables. Until now
all six were hard-coded in the Fortran, so every run used Wallin's values.

**The normalization fix:** SPAM divided the unit galaxy by a fixed 7.13333, the sum of the default masses
(`m_halo + h_disk + m_bulge` = 5.8 + 1.0 + 0.3333). So raising `m_halo` raised the galaxy's *total* mass, on top
of `mass1`/`mass2`. And the table's true total is 7.101 (the halo's tail past r = 2 is cut off), so even the
default galaxy is 0.9955, not 1.

| `normalization` | Divides by | Unit galaxy's mass |
|---|---|---|
| `"total"` (default in `sim`) | the table's actual total | always 1: the six values only shape it |
| `"wallin"` (the original, default in `jspam`) | 7.13333 | 0.9955 at the defaults, and changes with the masses |

**How it connects:**
- **Below, in JSPAM:** `df_module`'s `init_distribution` reads the six values from module variables (Wallin's
  values by default) and divides by `norm` instead of 7.13333. `simr_interface` gets `set_mass_model` (like
  `set_seed`) and `mass_profile`.
- **Next (5b, later):** two unit galaxies, one per galaxy.
- **Step 6:** timing and orbits. The start time comes from an orbit that uses these same tables.

The full, current demo is `simulator_tour.ipynb`.

**To run it** (from this repo's root, in bash or PowerShell), then open the `127.0.0.1` link it prints:
```bash
docker run --rm -it -p 8888:8888 -v "${PWD}:/gtc" -w /gtc -e UV_PROJECT_ENVIRONMENT=/tmp/venv \
    ogdenm12/gtc-base-image:v2 uv run --group notebook jupyter lab --ip=0.0.0.0 --allow-root --no-browser
```
- `-v "${PWD}:/gtc"` mounts this repo into the container, so edits on Windows show up immediately.
- `UV_PROJECT_ENVIRONMENT=/tmp/venv` keeps the Linux virtual environment inside the container, out of the repo.
  It's rebuilt each time the container starts (a minute or two, mostly compiling SPAM).
- Our package is installed in editable mode. After editing a `.py` file, restart the kernel to pick it up.

## What changed
**JSPAM, `fortran/df_module.f90`:** the six local constants became module variables, with Wallin's values as
defaults, and `init_distribution` reads them. The fixed divisor became `norm`:
```fortran
! module variables (defaults: exactly what init_distribution used to set)
real (kind=8) :: mm_lnl = 0.001d0, mm_rchalo = 10.0d0, mm_mhalo = 5.8d0   ! (one per line in the file)
real (kind=8) :: mm_mbulge = 0.3333, mm_hbulge = 0.2, mm_hdisk = 1.0d0
integer (kind=4) :: mass_normalization = 0

! in init_distribution
mhalo = mm_mhalo   ! SIMR: was 5.8d0   (the same for lnl, rchalo, hdisk, hbulge, mbulge)
...
norm = 7.13333d0
if (mass_normalization == 1) norm = mass_total(nmax)
do j = 1, ntotal
  mass_total(j) = mass_total(j) / norm   ! was / 7.13333d0, and the same for the other 12 tables
  ...
```
**JSPAM, `fortran/simr_interface.f90`:**
```fortran
subroutine set_mass_model(values, normalization)   ! values: lnl, r_scale, m_halo, m_bulge, h_bulge, h_disk
subroutine mass_profile(nrows, radius, m_halo, m_disk, m_bulge, m_total, force)   ! builds and returns the tables
```
**JSPAM, `src/jspam/__init__.py`:** `MASS_MODEL_DEFAULTS`, `NORMALIZATIONS`, and `mass_model=` / `normalization=`
on `run_collision`, `initial_state` and the new `mass_profile`. They're set before every run, so one run's model
can't leak into the next.

**`sim.py`:** seven new `SimulatorConfig` fields, with checks, and `sim.mass_profile(config)`.
```python
lnl: float = 0.001
r_scale: float = 10.0
m_halo: float = 5.8
m_bulge: float = 0.3333
h_bulge: float = 0.2
h_disk: float = 1.0
normalization: str = "total"
```
**Tests:** each value is checked to change the part of the galaxy it names (so a swap like the old code's
`m_bulge`/`h_disk` one would fail), plus both normalizations, `lnl` on a close encounter, and no leaking between runs.

In [ ]:
%matplotlib inline
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints.simr import sim

MODEL_STRING = "-9.93853,-4.5805,15.43348,-1.54025,-2.97899,4.69234,34.41671,29.49607,2.97913,4.2501,301.0,35.5,310.80745,321.98757,0.3,0.3,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0,0,0.24008,-2.41999,5.62715,0.0,1.0,0.0,1.0,0.0,0.0"
params = sim.parse_model_string(MODEL_STRING)
config = sim.SimulatorConfig()
WALLIN = replace(config, normalization="wallin")

## 1. The unit galaxy
Enclosed mass of each component, and the force a particle feels. The halo, disk and bulge tables end at r = 2,
so the mass stops growing there. One SPAM distance unit is about 15 kpc.

In [ ]:
profile = sim.mass_profile(config)
r = profile["radius"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name in ["halo", "disk", "bulge", "total"]:
    axes[0].plot(r, profile[name], label=name)
axes[0].set_xlim(0, 3)
axes[0].set_xlabel("radius (SPAM units)")
axes[0].set_ylabel("enclosed mass (galaxy mass = 1)")
axes[0].legend()
axes[1].plot(r, profile["force"], label="SPAM's force")
axes[1].plot(r[1:], profile["total"][1:] / r[1:] ** 2, "--", label="enclosed mass / r²")
axes[1].set_xlim(0, 10)
axes[1].set_yscale("log")
axes[1].set_xlabel("radius (SPAM units)")
axes[1].legend()
plt.show()
for name in ["halo", "disk", "bulge", "total"]:
    print(f"{name:6s} {profile[name][-1]:.5f}")

The dashed line is plain gravity from the enclosed mass. SPAM's force matches it (within 0.2%) from r = 0.05 to
the table's end at r = 10. Inside r = 0.05 it's deliberately smoothed, so particles near the center don't get a
huge kick. **Past r = 10 the table ends, and SPAM keeps using its last row, so the force stops falling.** That's
Wallin's behavior, an open question in MDRN-1, and this step doesn't change it.

## 2. The normalization
The same three halo masses, divided both ways. With `"wallin"`, doubling `m_halo` nearly doubles the galaxy.
With `"total"` it stays 1, and only the share in the halo changes.

In [ ]:
print(f"{'m_halo':>7s}   {'wallin total':>12s}   {'total total':>11s}   {'halo share':>10s}")
for m_halo in [2.9, 5.8, 11.6]:
    wallin = sim.mass_profile(replace(WALLIN, m_halo=m_halo))
    total = sim.mass_profile(replace(config, m_halo=m_halo))
    print(f"{m_halo:7.1f}   {wallin['total'][-1]:12.4f}   {total['total'][-1]:11.4f}   {total['halo'][-1]:10.3f}")

## 3. What each value does
Each panel doubles (or halves) one value and shows the total enclosed mass, with `"total"` normalization so
every curve ends at 1. `lnl` isn't in the tables: it only sets dynamical friction (section 5).

In [ ]:
changes = {
    "m_halo": [2.9, 11.6], "m_bulge": [0.15, 0.6], "h_bulge": [0.1, 0.4], "h_disk": [0.5, 2.0], "r_scale": [5.0, 20.0],
}
base = sim.mass_profile(config)
fig, axes = plt.subplots(len(changes), 1, figsize=(12, 3 * len(changes)), sharex=True, sharey=True)
for ax, (name, values) in zip(axes, changes.items()):
    ax.plot(base["radius"], base["total"], "k", label=f"{getattr(config, name)} (default)")
    for value in values:
        p = sim.mass_profile(replace(config, **{name: value}))
        ax.plot(p["radius"], p["total"], label=str(value))
    ax.set_xlim(0, 2)
    ax.set_ylabel("enclosed mass")
    ax.set_title(name, loc="left")
    ax.legend(loc="lower right")
axes[-1].set_xlabel("radius (SPAM units)")
fig.tight_layout()
plt.show()

**`r_scale` is the odd one.** It's the halo's core radius, but the tables' radius grid is built from it too,
so it stretches every component, while SPAM keeps reading the tables at fixed distances. In force that shows up
as roughly (10 / `r_scale`)² outside the core, as MDRN-1 measured:

In [ ]:
base_force = sim.mass_profile(config)["force"]
row = np.searchsorted(r, 5.0)  # r = 5
for r_scale in [5.0, 10.0, 20.0]:
    force = sim.mass_profile(replace(config, r_scale=r_scale))["force"]
    print(f"r_scale {r_scale:5.1f}: force at r = 5 is {force[row] / base_force[row]:.3f} x the default"
          f"   (10 / r_scale)^2 = {(10 / r_scale) ** 2:.3f}")

## 4. Effect on a run
Raising `m_halo` to 11.6: with `"wallin"` the galaxies get much heavier, so the encounter changes a lot. With
`"total"` they keep their masses, and only the distribution changes. Same seed in every panel.

In [ ]:
small = replace(config, n1=1500, n2=1500)
runs = {
    "total, defaults": small,
    "total, m_halo 11.6": replace(small, m_halo=11.6),
    "wallin, m_halo 11.6": replace(small, m_halo=11.6, normalization="wallin"),
}
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharex=True, sharey=True)
for ax, (title, cfg) in zip(axes, runs.items()):
    final = sim.simulate(params, cfg, seed=1)
    ax.scatter(final[:cfg.n1, 0], final[:cfg.n1, 1], s=1)
    ax.scatter(final[cfg.n1:-1, 0], final[cfg.n1:-1, 1], s=1)
    ax.set_title(title)
    ax.set_aspect("equal")
plt.show()

## 5. Dynamical friction (`lnl`)
Friction slows the secondary as it moves through the primary's halo, but the halo's density table ends at
r = 2, so it only acts when the secondary passes inside that. The SDSS model above never does. Line 3 of
`data.txt` does:

In [ ]:
close = np.array([1.08119, -1.66798, -2.06826, -0.18438, -0.39693, -0.02919, 0.51654, 0.36192, 0.94043, 0.43877,
                  12.97415, 135.11902, 97.54035, 305.01225, 0.3, 0.3, 0, 0, 0, 0, 0, 0])
tiny = replace(config, n1=5, n2=5)
for lnl in [0.0, 0.001, 0.1, 0.5]:
    end = sim.simulate(close, replace(tiny, lnl=lnl), seed=1)[-1]
    print(f"lnl {lnl:5.3f}: secondary ends at {end[:3].round(4)}")